# Problem statement: census income DE pipeline

## Context

Raw census-style records land from **batch, web, app, and CRM** (`source_system`) with inconsistent column casing, stray symbols in numeric fields (`$`, `%`, `k`), mixed `event_time` formats, missing categoricals, and occasional corrupt `age` values. The file is a **donation / census hybrid** used for analytics and modeling.

## Downstream contract (ML notebook)

The ML pipeline reads **`workshop.default.donation_data_v1`**, keeps rows with **`income` in `('<=50k', '>50k')`**, then:

- Casts **numeric** columns to double: `age`, `education-num`, `capital-gain`, `capital-loss`, `hours-per-week` (stripping non-numeric noise first).
- Uses **categorical** columns: `workclass`, `education_level`, `marital-status`, `occupation`, `relationship`, `race`, `sex`, `native-country` (filled with `"Unknown"` where null).
- Imputes numeric nulls with **approximate median**, drops **`random_flag`**, and drops **`event_time_std`** only at sklearn boundary (column still useful in the lake for auditing).

So **DE must** deliver a stable schema, normalized labels, and string forms that survive `regexp_replace` + cast in ML without `NumberFormatException`.

## DE objectives

1. **Ingest** messy CSV from a UC volume or an equivalent path.
2. **Normalize** column names (lower + trim) and align with ML column names.
3. **Repair / reject** bad rows (e.g. `age` like `ERROR_%`).
4. **Standardize** `event_time` into **`event_time_std`** via multiple `try_to_timestamp` patterns.
5. **Normalize `income`** to lowercase forms expected by ML (`<=50k`, `>50k`).
6. **Sanitize** all ML numeric columns in DE (same regex idea as ML) to reduce cast failures.
7. **Trim** key categorical strings and coalesce nulls where appropriate before deduplication.
8. **Deduplicate**, drop raw `event_time`, **validate** row-level rules and optional outlier flags.
9. **Publish** to Delta **`workshop.default.donation_data_v1`** (overwrite) for training and reporting.

## Success criteria

- Table builds **idempotently** (overwrite mode).
- **`income`** distribution is queryable; values match ML `isin` filter after normalization.
- ML numeric columns parse as doubles after DE (no stray `$` / `%` in persisted strings where avoidable).
- **`event_time_std`** populated where source timestamp is parseable; nulls acceptable but measurable.


In [0]:
# Use volume path on Databricks; use file:// path for local Spark with census_data_raw.csv
path ="/Users/sbarde/Downloads/census_data_raw.csv"


df = spark.read.csv(path, header=True, inferSchema=True)
display(df)

: 

## Processing Steps

In [0]:
from pyspark.sql import functions as F

df_stage = df.select([F.lower(F.trim(F.col(c))).alias(c.lower().strip()) for c in df.columns])

: 

In [0]:
df_stage = df_stage.filter(~F.col("age").cast("string").like("ERROR_%"))

In [0]:
df_stage = df_stage.withColumn("age", F.regexp_replace(F.col("age"), "[^0-9]", ""))

In [0]:
df_stage = df_stage.withColumn("workclass", F.coalesce(F.col("workclass"), F.lit("Unknown")))

In [0]:
df_stage = df_stage.withColumn("workclass", F.trim(F.col("workclass")))

In [0]:
df_final = df_stage.withColumn(
    "event_time_std",
    F.coalesce(
        F.try_to_timestamp(F.col("event_time"), F.lit("yyyy-MM-dd")),
        F.try_to_timestamp(F.col("event_time"), F.lit("dd/MM/yyyy")),
        F.try_to_timestamp(F.col("event_time"), F.lit("MM-dd-yyyy HH:mm:ss"))
    )
)

In [0]:
df_clean = df_final.dropDuplicates()

In [0]:
df_clean = df_clean.drop("event_time")

## ML contract and advanced cleansing (complex examples)

The cells below go **after** raw `event_time` is dropped and **`df_clean`** is the working dataset. They encode the **implicit contract** with `ML_modelling.ipynb`: normalized `income`, ML numeric columns stripped of junk characters, trimmed categoricals, a **DQ aggregate**, **z-score by partition** (window), a **row-level consistency flag** between `education_level` and `education-num`, and **UDF / pandas UDF** patterns where Python logic is clearer than pure `Column` expressions (use UDFs sparingly on large data; `pandas_udf` is vectorized per batch).

In [ ]:
# Columns the ML notebook consumes (keep DE outputs aligned with that contract)
ML_NUM_COLS = ["age", "education-num", "capital-gain", "capital-loss", "hours-per-week"]
ML_CAT_COLS = [
    "workclass",
    "education_level",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native-country",
]

# 1) Income labels: ML uses F.col("income").isin("<=50k", ">50k") — force lowercase + trim
df_clean = df_clean.withColumn("income", F.lower(F.trim(F.col("income"))))

# 2) Numeric hygiene: strip $, %, k, commas, etc. (same intent as ML regexp_replace on string form)
for c in ML_NUM_COLS:
    if c in df_clean.columns:
        df_clean = df_clean.withColumn(
            c,
            F.regexp_replace(F.col(c).cast("string"), "[^0-9.]", ""),
        )

# 3) Categorical trim (null-safe) — reduces duplicate categories that differ only by spaces
for c in ML_CAT_COLS:
    if c in df_clean.columns:
        df_clean = df_clean.withColumn(c, F.trim(F.col(c)))

# 4) Optional modeling-ready filter (uncomment if the Delta table should only contain labeled rows)
# df_clean = df_clean.filter(F.col("income").isin("<=50k", ">50k"))

In [ ]:
# UDF examples (scalar udf + pandas UDF). Prefer Column APIs for simple transforms; use UDFs for messy text rules.
import re
import pandas as pd
from pyspark.sql.functions import pandas_udf, udf
from pyspark.sql.types import StringType

# --- 1) Scalar PySpark UDF: collapse repeated internal whitespace (e.g. "Adm  clerical") ---
@udf(StringType())
def collapse_whitespace(val):
    if val is None:
        return None
    t = str(val).strip()
    if not t:
        return None
    return re.sub(r"\s+", " ", t)


if "occupation" in df_clean.columns:
    df_clean = df_clean.withColumn("occupation", collapse_whitespace(F.col("occupation")))



In [ ]:
# Data-quality snapshot: null rates and label validity (complex aggregation pattern)
label_ok = F.col("income").isin("<=50k", ">50k")

dq_summary = df_clean.select(
    F.count(F.lit(1)).alias("row_cnt"),
    F.sum(F.when(F.col("income").isNull(), 1).otherwise(0)).alias("null_income"),
    F.sum(F.when(~label_ok & F.col("income").isNotNull(), 1).otherwise(0)).alias("unexpected_income"),
    F.sum(F.when(F.col("event_time_std").isNull(), 1).otherwise(0)).alias("null_event_time_std"),
    F.sum(F.when(F.col("age").cast("string") == "", 1).otherwise(0)).alias("blank_age_string"),
)

display(dq_summary)

In [ ]:
# Window example: age z-score within sex (detect extreme rows while keeping full grain)
from pyspark.sql import Window

w_sex = Window.partitionBy("sex")
age_d = F.col("age").cast("double")

df_clean = (
    df_clean.withColumn("_mu", F.avg(age_d).over(w_sex))
    .withColumn("_sig", F.stddev_pop(age_d).over(w_sex))
    .withColumn(
        "age_z_by_sex",
        (age_d - F.col("_mu")) / F.nullif(F.col("_sig"), F.lit(0.0)),
    )
    .drop("_mu", "_sig")
)

# Row-level rule: education text present but numeric years missing (cross-field DQ)
df_clean = df_clean.withColumn(
    "dq_edu_level_without_num",
    F.col("education_level").isNotNull()
    & (F.length(F.trim(F.col("education_level"))) > 0)
    & (
        F.col("education-num").isNull()
        | (F.length(F.trim(F.col("education-num").cast("string"))) == 0)
    ),
)

display(df_clean.select("sex", "age", "age_z_by_sex", "education_level", "education-num", "dq_edu_level_without_num").filter(F.col("dq_edu_level_without_num")).limit(20))

In [ ]:
# Complex analytics pattern: decade buckets without adding columns to df_clean
age_d = F.col("age").cast("double")

_age_grp = (
    df_clean.select((F.floor(age_d / 10) * 10).cast("int").alias("age_decade_start"))
    .groupBy("age_decade_start")
    .count()
    .orderBy("age_decade_start")
)
display(_age_grp)

In [ ]:
# Drop exploratory-only columns so sklearn does not pick up accidental extra features
_drop = [c for c in ("age_z_by_sex", "dq_edu_level_without_num") if c in df_clean.columns]
if _drop:
    df_clean = df_clean.drop(*_drop)

In [0]:
df_grouped = df_clean.groupBy("income").count()
display(df_grouped)

In [0]:
df_clean.write.format("delta").mode("overwrite").saveAsTable("workshop.default.donation_data_v1")